# Hypothesis 2 Testing

_If someone reads an increasing number of articles over a time period, they are more likely to subscribe to newsletters._

## Caveats

TKTK

## Clarification

- "articles" -> only look at page views of articles, i.e., `PAGE_IS_ARTICLE is True`.
- "increasing number of articles" -> we're interested in trajectory of count for each user/reader
- "more likely to subscribe to newsletters" -> we can keep 0 or 1 as the boolean response

Since calculating the correlation between the trajectory of a variable and a binary outcome is not as straightforward as doing so between said variable and said outcome, for this hypothesis, before diving into correlations, let's do a visual analysis first.

One way of doing this is to create two subpopulations of readers: subscribers and non-subscribers. For each subpopulation, we can plot out the average trajectory of article count over time (after rebasing a user's events' timestamps to the timestamp of their first event). 

If our hypothesis holds true, we should see an upward average trajectory among subscribers. The same plot for non-subscribers serves as a point of reference.

## Preprocessing

In [ ]:
from enum import auto

import pandas as pd
from helpers.pathlib import get_path_site_checkpoint

from ata_pipeline1.helpers.enums import FieldNew, FieldSnowplow, StrEnum
from ata_pipeline1.helpers.urllib import append_slash
from ata_pipeline1.site import AFRO_LA, DALLAS_FREE_PRESS, OPEN_VALLEJO, THE_19TH

In [ ]:
# Nov. 3, 2022 to Jan. 25, 2022 (inclusively) a.k.a 12 weeks
CSV_PREFIX = "221103-230125"

In [ ]:
# INSTRUCTION: If there's a new field you'd like to add, add its name to this enum
class FieldTemp(StrEnum):
    ARTICLE_COUNT_PERIOD_INDEX = auto()
    ARTICLE_COUNT_PERIOD_START_DATE = auto()
    #     ARTICLE_COUNT_WINDOW_END_DATE = auto()
    #     DAYS_FROM_EARLIEST_TO_SUBMISSION = auto()
    DERIVED_TSTAMP_DATE = auto()
    EARLIEST_EVENT_TSTAMP = auto()
    EARLIEST_EVENT_DATE = auto()
    FIRST_PERIOD_TSTAMP = auto()
    LAST_EVENT_TSTAMP = auto()
    LAST_EVENT_DATE = auto()
    LAST_SUBMISSION_TSTAMP = auto()
    NUM_PERIODS = auto()
    NUM_ARTICLES_READ = auto()
    SUBMITTED = auto()

In [ ]:
# Read data from CHECKPOINT 7
df_afla = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, AFRO_LA.name))
df_dfp = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, DALLAS_FREE_PRESS.name))
df_ov = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, OPEN_VALLEJO.name))
df_19 = pd.read_pickle(get_path_site_checkpoint(CSV_PREFIX, 7, THE_19TH.name))

In [ ]:
# Filter to page views of articles and page views where a newsletter-form submission was recorded
def get_articles_and_submissions_dfs(df, site_name):
    df_articles = df.query(f"{FieldNew.PAGE_IS_ARTICLE} " + f"and {FieldNew.DWELL_SECS} >= 10 ")
    df_submissions = df.query(FieldNew.FORM_SUBMIT_IS_NEWSLETTER)
    print(f"Site: {site_name}")
    print(f"Found {df_articles.shape[0]} article views that last more than 10 seconds")
    print(f"Found {df_submissions.shape[0]} page views where a newsletter form is submitted")
    print("\n")
    return df_articles, df_submissions


df_afla_articles, df_afla_submissions = get_articles_and_submissions_dfs(df_afla, AFRO_LA.name)
df_dfp_articles, df_dfp_submissions = get_articles_and_submissions_dfs(df_dfp, DALLAS_FREE_PRESS.name)
df_ov_articles, df_ov_submissions = get_articles_and_submissions_dfs(df_ov, OPEN_VALLEJO.name)
df_19_articles, df_19_submissions = get_articles_and_submissions_dfs(df_19, THE_19TH.name)

In [ ]:
# Make article URLs of uniform format so that nunique() later returns accurate results
def make_uniform_urlpaths(df_articles):
    df_articles = df_articles.copy()
    df_articles[FieldSnowplow.PAGE_URLPATH] = df_articles[FieldSnowplow.PAGE_URLPATH].apply(append_slash)
    return df_articles


df_afla_articles = make_uniform_urlpaths(df_afla_articles)
df_dfp_articles = make_uniform_urlpaths(df_dfp_articles)
df_ov_articles = make_uniform_urlpaths(df_ov_articles)
df_19_articles = make_uniform_urlpaths(df_19_articles)

### Get all unique users

In [ ]:
# Group events by user and get the date of earliest event for each user
def group_events(df, site_name):
    df_grouped = df.groupby(FieldSnowplow.DOMAIN_USERID).aggregate(
        **{
            FieldTemp.EARLIEST_EVENT_TSTAMP: pd.NamedAgg(column=FieldSnowplow.DERIVED_TSTAMP, aggfunc="min"),
            FieldTemp.LAST_EVENT_TSTAMP: pd.NamedAgg(column=FieldSnowplow.DERIVED_TSTAMP, aggfunc="max"),
            #             FieldTemp.DAYS_ON_SITE: pd.NamedAgg(column=FieldSnowplow.DERIVED_TSTAMP, aggfunc=lambda x: (x.max() - x.min()).total_seconds() / 60 / 60 / 24),
        }
    )
    df_grouped[FieldTemp.EARLIEST_EVENT_DATE] = df_grouped[FieldTemp.EARLIEST_EVENT_TSTAMP].dt.date
    df_grouped[FieldTemp.LAST_EVENT_DATE] = df_grouped[FieldTemp.LAST_EVENT_TSTAMP].dt.date
    print(f"Site: {site_name}")
    print(f"Found {df_grouped.shape[0]} unique readers")
    #     print(f"Minimum days on site: {df_grouped[FieldTemp.DAYS_ON_SITE].min():.3f}")
    #     print(f"Median days on site: {df_grouped[FieldTemp.DAYS_ON_SITE].median():.3f}")
    #     print(f"Maximum days on site: {df_grouped[FieldTemp.DAYS_ON_SITE].max():.3f}")
    print("\n")
    return df_grouped


df_afla_users = group_events(df_afla, AFRO_LA.name)
df_dfp_users = group_events(df_dfp, DALLAS_FREE_PRESS.name)
df_ov_users = group_events(df_ov, OPEN_VALLEJO.name)
df_19_users = group_events(df_19, THE_19TH.name)

### Get all unique users who sign up (a.k.a. subscribers)

In [ ]:
# Group events by user. If a user has more than one event, take the most recent one to ensure X is
# as big as it can be
def group_submissions(df_submissions, df_users, site_name):
    df_grouped = (
        df_submissions.groupby(FieldSnowplow.DOMAIN_USERID)
        .aggregate({FieldSnowplow.DERIVED_TSTAMP: "max"})
        .rename(columns={FieldSnowplow.DERIVED_TSTAMP: FieldTemp.LAST_SUBMISSION_TSTAMP})
        .merge(df_users[FieldTemp.EARLIEST_EVENT_TSTAMP], left_index=True, right_index=True)
    )
    df_grouped[FieldTemp.SUBMITTED] = True  # True
    #     df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION] = (
    #         (df_grouped[FieldTemp.LAST_SUBMISSION_TSTAMP] - df_grouped[FieldTemp.EARLIEST_EVENT_TSTAMP]).dt.total_seconds()
    #         / 60
    #         / 60
    #         / 24
    #     )
    print(f"Site: {site_name}")
    print(f"Found {df_grouped.shape[0]} readers who signed up for newsletter")
    #     print(
    #         f"Smallest window between first event and newsletter sign-up: {df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION].min():.3} days"
    #     )
    #     print(
    #         f"Greatest window between first event and newsletter sign-up: {df_grouped[FieldTemp.DAYS_FROM_EARLIEST_TO_SUBMISSION].max():.3} days"
    #     )
    print("\n")
    return df_grouped


df_afla_users_submission = group_submissions(df_afla_submissions, df_afla_users, AFRO_LA.name)
df_dfp_users_submission = group_submissions(df_dfp_submissions, df_dfp_users, DALLAS_FREE_PRESS.name)
df_ov_users_submission = group_submissions(df_ov_submissions, df_ov_users, OPEN_VALLEJO.name)
df_19_users_submission = group_submissions(df_19_submissions, df_19_users, THE_19TH.name)

### Group article views by time period

In [ ]:
# INSTRUCTIONS: Specify period length in days
PERIOD_LENGTH_DAYS = 1

In [ ]:
def add_period_counts_users(df_users):
    return df_users.assign(
        **{
            FieldTemp.NUM_PERIODS: lambda x: (x[FieldTemp.LAST_EVENT_DATE] - x[FieldTemp.EARLIEST_EVENT_DATE]).dt.days
            // PERIOD_LENGTH_DAYS
        }
    )


df_afla_users = add_period_counts_users(df_afla_users)
df_dfp_users = add_period_counts_users(df_dfp_users)
df_ov_users = add_period_counts_users(df_ov_users)
df_19_users = add_period_counts_users(df_19_users)

In [ ]:
def group_article_events(df_articles, df_users, include_zero_count_periods: bool = False):
    def create_period_indices(row):
        derived_tstamp_date = row[FieldSnowplow.DERIVED_TSTAMP].date()
        domain_userid = row[FieldSnowplow.DOMAIN_USERID]
        user_first_event_date = df_users.loc[domain_userid][FieldTemp.EARLIEST_EVENT_DATE]
        row[FieldTemp.ARTICLE_COUNT_PERIOD_INDEX] = (
            derived_tstamp_date - user_first_event_date
        ).days // PERIOD_LENGTH_DAYS
        return row

    df_grouped = (
        df_articles.copy()
        .apply(create_period_indices, axis=1)
        .groupby([FieldSnowplow.DOMAIN_USERID, FieldTemp.ARTICLE_COUNT_PERIOD_INDEX])[FieldSnowplow.PAGE_URLPATH]
        .nunique()
        .rename(FieldTemp.NUM_ARTICLES_READ)
    )

    df_grouped = pd.DataFrame(df_grouped)

    if include_zero_count_periods:
        all_periods = df_users[FieldTemp.NUM_PERIODS].map(lambda n: range(0, n + 1, 1)).explode()
        df_all_periods = (
            pd.DataFrame({FieldTemp.ARTICLE_COUNT_PERIOD_INDEX: all_periods, FieldTemp.NUM_ARTICLES_READ: 0})
            .reset_index()
            .set_index([FieldSnowplow.DOMAIN_USERID, FieldTemp.ARTICLE_COUNT_PERIOD_INDEX])
        )
        df_grouped = (df_all_periods + df_grouped).fillna(0).astype(int)

    return df_grouped

In [ ]:
df_afla_users_article = group_article_events(df_afla_articles, df_afla_users)
df_dfp_users_article = group_article_events(df_dfp_articles, df_dfp_users)
df_ov_users_article = group_article_events(df_ov_articles, df_ov_users)
df_19_users_article = group_article_events(df_19_articles, df_19_users)

### Split users into subscribers and non-subscribers